# Apache Spark in Colab: the BlacFraidi of MediaMas 🛒

In Sessions 1 and 2 you worked with **Hadoop**: HDFS and **MapReduce**. Today you meet its successor, **Apache Spark** (we use the latest release, **4.3.0**), which keeps the data **in memory** between steps and lets you express a whole pipeline in a few lines.

**The story.** You have just joined the data team of **MediaMas**, a Spanish chain of department stores. Last week was **BlacFraidi**, the biggest sales campaign of the year, and the management wants answers:
*when did we sell the most? which channel and which products work best? who returns what?* The raw data is the file **`TrainBF`**: **250 000 orders** (20-30 November 2026, online, in the app and in the stores).

| Part | What you will do |
|---|---|
| 0 | 🔁 Install Java + PySpark and start a `SparkSession` (ready-made cells) |
| 1 | **RDDs**: transformations, actions, lazy evaluation, partitions — and a 🎯 *Monte Carlo* estimate of **π** you can see |
| 2 | **DataFrames & SQL**: explore `TrainBF`, clean it, aggregate it, read the Catalyst plan |
| 3 | 🌊 **Structured Streaming**: the campaign *live*, files arriving little by little |
| 4 | 🌐 **Spark UI** with ngrok *(optional)* |
| 5 | 🔌 **Spark Connect**: the new client/server architecture |
| 6 | 🧩 **Final exercise**: your report for the management of MediaMas |

**How it works.** Same as before: every step has a *goal* and the *expected result*, the code cells are **empty: you write the code**
(except Part 0 and the cells marked 🎨 or *ready-made*), and every question has a box below it where **you write the answer**. Run the cells **in order**.

> ⚠️ A Colab runtime is temporary: if it is recycled you lose the variables and the files. Run Part 0 again and continue from where you were.
> 🐼 *Next session: pandas, and how its code scales up to Spark.* Today you only use pandas to draw a chart.

# Part 0 – Install Spark (ready-made)

Spark runs on the JVM, so we need Java 17. PySpark (the Python API) is installed with `pip` and **already contains Spark 4.3.0**; the `connect` extra adds the libraries that Part 5 needs, so there is no manual download.
Run the three cells below, in order (≈ 2 min). The last one creates the `SparkSession`: the *entry point* of every Spark program (`spark`), and its `SparkContext` (`sc`), used by the RDD API.
We run Spark in **local mode** with `local[*]`: one JVM that uses all the CPU cores of the Colab machine as if they were workers.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
pip uninstall -y -q dataproc-spark-connect 2>/dev/null || true
pip install -q "pyspark[connect]==4.3.0" pyngrok
java -version 2>&1 | head -n 1

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("MediaMas")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| cores (default parallelism):", sc.defaultParallelism)

# Part 1 – RDDs: the heart of Spark

An **RDD** (*Resilient Distributed Dataset*) is a collection split into **partitions** that live on different workers. You work on it with two kinds of operations:
**transformations** (`map`, `filter`, `flatMap`, `reduceByKey`...) build a *new* RDD and are **lazy** (nothing is computed yet), and **actions** (`count`, `reduce`, `collect`, `take`...) trigger the real computation.

## 1.1 Your first RDD

Create an RDD with the numbers **1 to 1 000 000** (`sc.parallelize`) and answer two things: how many **partitions** it has, and the **sum of the squares of the even numbers** (use `filter`, `map` and `reduce`).

<details><summary>💡 Hint</summary>

`sc.parallelize(range(1, 1000001))`, `rdd.getNumPartitions()`. The pipeline is `rdd.filter(...).map(...).reduce(lambda a, b: a + b)`.

</details>

✅ **Expected:** Partitions = number of cores of the machine (2 on a standard Colab) and the sum is `166667166667000000`. `filter` and `map` are transformations (lazy); `reduce` is the action that made Spark run the job.

In [ ]:
# Your code here

❓ **Question:** Which of the operations you used are transformations and which one is the action? Where did the 'work' happen?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.2 Laziness and caching

Build an RDD that is **expensive to compute** (for example, `sc.parallelize(range(3_000_000))` with a `map` that calls a small function doing some maths) and time (`time.time()`) **three** things:
(a) creating the transformation, (b) the first `count()`, (c) a second `count()`. Then `.cache()` the RDD, run `count()` twice more and compare the times.

<details><summary>💡 Hint</summary>

Take `t0 = time.time()` before and print `time.time() - t0` after each step. `cache()` is itself lazy: the first action after it fills the cache.

</details>

✅ **Expected:** (a) ≈ 0 s; (b), (c) and (d) of the same order (a few seconds: without cache Spark *recomputes* the whole pipeline on every action; (b) also pays the start-up of the Python workers); (e) clearly the fastest (≈ 1 s: the data is already in memory).

In [ ]:
# Your code here

❓ **Question:** Why is (a) instantaneous? Why is the second `count()` after `cache()` much faster than the first one?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 🎯 1.3 Showcase: estimate **π** by throwing darts

Throw random points in the square `[0,1] × [0,1]`. The fraction that falls **inside the quarter circle** `x² + y² < 1` is the ratio of the areas: **π/4**. So `π ≈ 4 × inside / total`.
Perfect for Spark: every point is independent, so the work can be split among all the partitions (it is *embarrassingly parallel*).

Write the Spark job (name the function `inside`, the next cell uses it): generate **N = 2 000 000** points with `sc.parallelize(range(N))`, keep with `filter` those that fall inside the circle and count them.

In [ ]:
# Your code here

✅ **Expected:** something like `π ≈ 3.14...` (the third decimal changes on every run: it is random).

<details><summary>💡 Hint</summary>

Define a function `inside(_)` that draws `x, y = random.random(), random.random()` and returns `x*x + y*y < 1`; then `sc.parallelize(range(N)).filter(inside).count()`.

</details>

### 🎨 Now *see* it

Ready-made cell (just run it). Left: 5 000 darts, **green** inside the circle and **red** outside. Right: how the estimate **converges** to π when you throw more darts (it uses Spark for every N).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(7)
x, y = rng.random(5000), rng.random(5000)
ins = x * x + y * y < 1

Ns = [10**k for k in range(2, 7)]              # 100 ... 1 000 000 darts
est = [4 * sc.parallelize(range(n)).filter(inside).count() / n for n in Ns]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5.2))
a.scatter(x[ins], y[ins], s=7, c="#2ca02c", label="inside")
a.scatter(x[~ins], y[~ins], s=7, c="#d62728", label="outside")
t = np.linspace(0, np.pi / 2, 200); a.plot(np.cos(t), np.sin(t), "k", lw=2)
a.set_aspect("equal"); a.set_title("5 000 darts: %.0f%% inside -> π ≈ %.3f" % (100 * ins.mean(), 4 * ins.mean()))
a.legend(loc="lower left")
b.semilogx(Ns, est, "o-", lw=2, color="#1f77b4", label="Spark estimate")
b.axhline(np.pi, color="k", ls="--", label="π")
b.set_xlabel("number of darts N"); b.set_ylabel("estimate of π"); b.set_title("More darts, better estimate"); b.legend()
plt.tight_layout(); plt.show()
print({n: round(float(e), 4) for n, e in zip(Ns, est)})

❓ **Question:** Why does the estimate fluctuate a lot with N = 100 and hardly at all with N = 1 000 000? How would you make the job run on more workers without changing the code of `inside`?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 2 – DataFrames and SQL: first look at `TrainBF`

RDDs are flexible but Spark knows nothing about what is *inside* them. A **DataFrame** is a distributed table with **named columns and types**: Spark can then optimise your query (the **Catalyst** optimiser) and you can even write it in **SQL**.
Each row of `TrainBF` is an **order line** of BlacFraidi: date and hour, customer (gender, age range, city), channel (`Online`, `App`, `Store`), product and category, unit price, quantity, discount, payment method, customer `rating` (1-5), whether it was `returned` and the `delivery_days` (online only).

## 2.1 Load the data

Download `https://github.com/AntSanVar/DistributedSystemData/raw/refs/heads/main/Session03/TrainBF.zip`, unzip it (you get `TrainBF.csv`) and read it in a DataFrame `df` with `spark.read.csv`, using the **header** and letting Spark **infer the schema**. Print the schema, the number of rows and the first 5 rows.

<details><summary>💡 Hint</summary>

`!wget -q` and `!unzip -o -q`; `spark.read.csv('TrainBF.csv', header=True, inferSchema=True)`; `df.printSchema()`, `df.count()`, `df.show(5)`.

</details>

✅ **Expected:** A schema with 18 columns (`order_id: integer`, `order_date: date`, `unit_price: double`, `rating: integer`, `delivery_days: integer`...), `rows: 250000` and 5 sample rows. `inferSchema` makes Spark read the file once to guess the types (a *job* runs); the rest is lazy.

In [ ]:
# Your code here

❓ **Question:** What type did Spark infer for `order_date`, `unit_price` and `rating`? Is reading the CSV a transformation or an action? (Look at what happened when you ran the cell.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.2 Missing values

Real data has holes. Count, **for every column**, how many rows are `null`, with a single `select` (use a list comprehension over `df.columns` with `F.count(F.when(F.col(c).isNull(), c))`).

<details><summary>💡 Hint</summary>

`from pyspark.sql import functions as F`; `df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()`.

</details>

✅ **Expected:** Only `rating` (≈ 35 % of the rows: many customers do not rate) and `delivery_days` (≈ 31 %: in-store purchases have no delivery) have nulls. They are not errors: the data is *missing by design*.

In [ ]:
# Your code here

❓ **Question:** Which columns have missing values and why? (Think about what the business means: do the nulls mean 'error'?)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.3 A new column: the money we really earned

The list price is not what the customer paid. Add a column `revenue` = `unit_price × quantity × (1 − discount_pct / 100)`, **rounded to 2 decimals** (`withColumn`). Show 5 rows with `order_id`, `category`, `unit_price`, `quantity`, `discount_pct` and `revenue`, and the **total revenue of the campaign**.

<details><summary>💡 Hint</summary>

`df = df.withColumn('revenue', F.round(F.col('unit_price') * F.col('quantity') * (1 - F.col('discount_pct') / 100), 2))`; the total is `df.agg(F.sum('revenue')).show()`.

</details>

✅ **Expected:** 5 rows with the new column and a `total_revenue` of about **25.5 million €** (`25,525,177.83`). `withColumn` is a transformation (lazy) and DataFrames are **immutable**: it returns a *new* DataFrame, which we assign back to `df`.

In [ ]:
# Your code here

❓ **Question:** Is `withColumn` an action or a transformation? Did the DataFrame `df` change, or did you get a new one?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.4 Channels and categories

How much did **each channel** sell? For each `channel` compute the number of order lines (`orders`), the units sold (`units`), the total `revenue` (rounded) and the average `discount` (rounded to 1 decimal), sorted by revenue (descending). Then do the same for each `category`.

<details><summary>💡 Hint</summary>

`groupBy('channel').agg(F.count('*').alias('orders'), F.sum('quantity').alias('units'), F.round(F.sum('revenue')).cast('long').alias('revenue'), F.round(F.avg('discount_pct'), 1).alias('discount')).orderBy(F.desc('revenue'))`. For the categories change the column.

</details>

✅ **Expected:** Three rows for the channels: `Online` (≈ 13.2 M€ in 130 156 order lines), `Store` (≈ 8.0 M€) and `App` (≈ 4.4 M€); and eight for the categories: `Electronics` first (≈ 17.6 M€, 22 % average discount), while `Fashion` has the highest discount (30 %) but earns far less.

In [ ]:
# Your code here

❓ **Question:** Which channel brings most of the money? Which category? Does the category with the highest revenue also have the highest discount?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 2.5 The same in SQL, and the plan

Register the DataFrame as a **temporary view** (`createOrReplaceTempView('orders')`) and write the **same query by channel in SQL** with `spark.sql(...)`. Then print the **physical plan** (`.explain()`) of the SQL query.

<details><summary>💡 Hint</summary>

`SELECT channel, COUNT(*) AS orders, SUM(quantity) AS units, CAST(ROUND(SUM(revenue)) AS BIGINT) AS revenue, ROUND(AVG(discount_pct), 1) AS discount FROM orders GROUP BY channel ORDER BY revenue DESC`.

</details>

✅ **Expected:** The same three rows as in 2.4. Plan, bottom to top: `FileScan csv` → partial `HashAggregate` (a *combiner* inside each partition) → `Exchange hashpartitioning` (**the shuffle**) → final `HashAggregate` → `Sort`. The DataFrame API gives the same plan: both go through Catalyst.

In [ ]:
# Your code here

❓ **Question:** Is the result identical to 2.4? Read the plan from the bottom to the top: where is the **shuffle** (the `Exchange`)? What is the `HashAggregate` that appears **twice**?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎨 When do people buy?

Ready-made cell: a **heat map** of the revenue by **day** and **hour** of the campaign. The `groupBy` runs in Spark; only the tiny result (11 days × 24 hours) goes to *pandas* with `toPandas()` to be drawn.

In [ ]:
import matplotlib.pyplot as plt

h = (df.groupBy("order_date", "order_hour").agg(F.sum("revenue").alias("rev")).toPandas()
       .pivot(index="order_date", columns="order_hour", values="rev").fillna(0).sort_index() / 1e3)
fig, ax = plt.subplots(figsize=(12, 4.6))
im = ax.imshow(h.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(len(h))); ax.set_yticklabels([str(d)[5:] for d in h.index])
ax.set_xlabel("hour of the day"); ax.set_ylabel("day (month-day)")
ax.set_title("MediaMas BlacFraidi: revenue (thousand €) by day and hour")
plt.colorbar(im, label="thousand €"); plt.tight_layout(); plt.show()

❓ **Question:** On which day does MediaMas earn most? At which hours? Is there anything strange at midnight (hour 0)? What would you tell the management?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 3 – 🌊 BlacFraidi *live*: Structured Streaming

So far you analysed the campaign **after** it had finished (*batch*). But the management of MediaMas wants to see the numbers **while the orders arrive**. Spark does it with **Structured Streaming**: you write the **same DataFrame query** and Spark runs it again and again on **small batches of new data** (*micro-batches*), keeping the result up to date.

We will simulate it with files: every **day** of the campaign is a CSV file that will *fall* into a folder, one by one, and Spark will process each new file as soon as it appears.

```
 feed/ (all the days)  --move one file at a time-->  stream_in/  --readStream-->  query  --writeStream-->  table "live_channel"
```

### 🎬 Prepare the feed (ready-made)

This cell splits `TrainBF.csv` into **one file per day** (11 files) in the folder `feed/` and creates the **empty** folder `stream_in/` that Spark will watch.

In [ ]:
import os, shutil, pandas as pd

shutil.rmtree("/content/feed", ignore_errors=True); shutil.rmtree("/content/stream_in", ignore_errors=True)
shutil.rmtree("/content/chk", ignore_errors=True)
os.makedirs("/content/feed"); os.makedirs("/content/stream_in")
for day, g in pd.read_csv("/content/TrainBF.csv", dtype=str, keep_default_na=False).groupby("order_date"):
    g.to_csv(f"/content/feed/orders_{day}.csv", index=False)
print(sorted(os.listdir("/content/feed")))
print("watched folder:", os.listdir("/content/stream_in"))

## 3.1 A streaming DataFrame

Create a **streaming DataFrame** `stream` that reads the CSV files of `/content/stream_in` as they arrive: `spark.readStream.schema(df.schema).option("header", True).option("maxFilesPerTrigger", 1).csv(...)`. Then add the `revenue` column **exactly as in step 2.3** and print `stream.isStreaming`.

<details><summary>💡 Hint</summary>

`spark.readStream` is the streaming twin of `spark.read`. `df.schema` is the schema of the DataFrame of Part 2. The `revenue` formula is the same one: `F.round(F.col('unit_price') * F.col('quantity') * (1 - F.col('discount_pct') / 100), 2)`.

</details>

✅ **Expected:** `is streaming: True`. A stream cannot infer the schema because the folder is empty (and the schema must not change between batches). `maxFilesPerTrigger = 1` makes every micro-batch read **one file** (one day). Nothing is processed yet: like any transformation, a stream is **lazy** until you *start* a query.

In [ ]:
# Your code here

❓ **Question:** Why do we have to give the schema (`df.schema`) instead of using `inferSchema`? What does `maxFilesPerTrigger = 1` do? Does the cell process any data yet?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 3.2 Start the streaming query

Write a query that adds up, **for each channel**, the number of orders and the revenue **since the beginning**, and keep the result in an **in-memory table** called `live_channel`:
`groupBy('channel').agg(...)` → `writeStream` → `outputMode('complete')` (the whole table is rewritten in every batch), `format('memory')`, `queryName('live_channel')`, `trigger(processingTime='2 seconds')` and `start()`.
Store the query in `query` and print its `status`.

<details><summary>💡 Hint</summary>

`stream.groupBy('channel').agg(F.count('*').alias('orders'), F.round(F.sum('revenue'), 2).alias('revenue')).writeStream.outputMode('complete').format('memory').queryName('live_channel').trigger(processingTime='2 seconds').start()`.

</details>

✅ **Expected:** A status like `{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}` (later `Waiting for data to arrive`): the query is **running in the background** (the cell finishes) and waits for files. `complete` rewrites the whole result table; `update` writes only the rows that changed; `append` only new rows (impossible for an aggregation without a watermark, because a total can still change).

In [ ]:
# Your code here

❓ **Question:** What is the difference between the output modes `complete`, `update` and `append`? Why is `complete` the right one here?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎬 Let the orders arrive (ready-made)

🎨 The cell moves **one daily file at a time** from `feed/` to `stream_in/`, waits until Spark has processed it (`query.processAllAvailable()`) and then **reads the table `live_channel`** with a normal SQL query and redraws the chart. Watch how the bars grow while the days go by (≈ 1 minute).

In [ ]:
import glob, time, os, shutil
import matplotlib.pyplot as plt
from IPython.display import clear_output

files = sorted(glob.glob("/content/feed/orders_*.csv"))
for i, f in enumerate(files, 1):
    shutil.move(f, "/content/stream_in/")            # a file "falls" in the watched folder
    query.processAllAvailable()                      # wait until Spark has processed the new file
    time.sleep(1)
    snap = spark.sql("SELECT channel, orders, revenue FROM live_channel ORDER BY channel").toPandas()
    clear_output(wait=True)
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.4))
    a.bar(snap["channel"], snap["revenue"] / 1e6, color=["#9467bd", "#ff7f0e", "#1f77b4"][:len(snap)])
    a.set_ylim(0, 14); a.set_ylabel("revenue (million €)"); a.set_title("Revenue so far")
    b.bar(snap["channel"], snap["orders"] / 1e3, color=["#9467bd", "#ff7f0e", "#1f77b4"][:len(snap)])
    b.set_ylim(0, 135); b.set_ylabel("order lines (thousands)"); b.set_title("Orders so far")
    fig.suptitle("BlacFraidi live - day %d of %d (%s)" % (i, len(files), os.path.basename(f)[7:17]))
    plt.tight_layout(); plt.show()

## 3.3 What did the stream do?

Look inside the query: print the **last progress** (`query.lastProgress`: batch id, rows read, duration) and the number of **micro-batches** that ran (`len(query.recentProgress)`). Then check the result: add up the `revenue` of the table `live_channel` with SQL and **compare it with the total of the batch analysis** of step 2.3 (25,525,177.83). Finally **stop** the query (`query.stop()`).

<details><summary>💡 Hint</summary>

`print(query.lastProgress['numInputRows'], query.lastProgress['batchId'])`; `spark.sql('SELECT ROUND(SUM(revenue), 2) FROM live_channel').show()`; `query.stop()`.

</details>

✅ **Expected:** The last batch read about 20 000 rows (one day: 19958), `recentProgress` keeps the 11 micro-batches and the totals are **250000 order lines and 25525177.83 €**: *exactly* the batch result (the same code, the same answer: that is the promise of Structured Streaming). `active streams: 0`. With the query stopped nothing reads the new file: it would wait in the folder until the query is **restarted** (with a `checkpointLocation` it would continue where it left).

In [ ]:
# Your code here

❓ **Question:** Does the streaming total match the batch total? What does that tell you about the two modes? What would happen to the table if a new file arrived now that the query is stopped?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 4 – 🌐 *(Optional)* See the Spark UI with ngrok

Every Spark application has a **web UI** (jobs, stages, SQL plans, executors) on port **4040**. Colab machines have no public address, so your browser cannot reach it. As in Session 1, **ngrok** creates a temporary public URL that forwards to a port of the Colab machine.

**Before running the cell** (once per Google account):
1. Create a **free account** at <https://dashboard.ngrok.com/signup> and copy your **authtoken** (*Your Authtoken* in the dashboard). It is personal: **never write it in the notebook** and never share it.
2. In Colab open the **🔑 Secrets** panel (left bar), add a secret called **`NGROK_TOKEN`** with your token and switch on *Notebook access*.

Then run the ready-made cell. If the secret does not exist the cell just says so and you can **skip this part**: nothing else depends on it.

In [ ]:
from pyngrok import ngrok
try:
    from google.colab import userdata
    ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    tunnel = ngrok.connect(4040, "http")
    print("Spark UI ->", tunnel.public_url)
except Exception as e:
    print("Optional part skipped:", type(e).__name__, "- add the NGROK_TOKEN secret (see above) and run this cell again.")

Open the URL (click *Visit Site* if ngrok shows a warning page) and explore: **Jobs** (one per action you ran), **Stages**, **SQL / DataFrame** (the plan of 2.5, with the numbers of rows of every node), **Structured Streaming** (the query of Part 3: batches, input rate, duration) and **Executors**.

❓ **Question:** *In the SQL tab open the query of step 2.5: how many rows enter the `Exchange`? Compare with the 250 000 rows that the `FileScan` reads. What does that tell you about the partial `HashAggregate`?*

✍️ **Your answer** *(double-click this cell and write below this line; write "skipped" if you did not do this part)*:

>

# Part 5 – 🔌 Spark Connect

Until now your notebook and Spark lived **in the same process** (`local[*]`): the Python code talked to a JVM that it had started itself. Since Spark 3.4, and as the recommended way in **Spark 4**, there is another architecture: **Spark Connect**.

```
 classic:   [ your Python + the Spark JVM ]  (one block, both die together)

 connect:   [ Python client ] ──gRPC (port 15002)──▶ [ Spark Connect server (JVM) ] ──▶ executors
            (thin: only a few libraries)              (plans are built here and run here)
```

The client only **builds the query plan** and sends it through **gRPC**; the **server** optimises and executes it and sends the results back (Arrow batches). Advantages: a *thin* client (no JVM, it can even run from a laptop or from other languages), a server that **survives** crashes of the client, and **remote** access to a shared cluster. Price: the client has no `SparkContext`, so **there is no RDD API**.

## 5.1 Stop the classic session

First free the memory: **stop** the classic session of Part 2 (the `SparkSession` called `spark`).

<details><summary>💡 Hint</summary>

`spark.stop()`.

</details>

✅ **Expected:** `classic session stopped`.

In [ ]:
# Your code here

## 5.2 Start the server

Start the **Spark Connect server** that comes with PySpark: the script `start-connect-server.sh` is in the `sbin` folder of the *pyspark* package (the folder of `pyspark.__file__`). Wait ~20 s and check with `ss -ltnp` that Java is **listening on port 15002** (Connect) and 4040 (the UI).

<details><summary>💡 Hint</summary>

A `%%bash` cell: `export SPARK_HOME=$(python -c "import pyspark,os;print(os.path.dirname(pyspark.__file__))")`, then `$SPARK_HOME/sbin/start-connect-server.sh`, `sleep 20` and `ss -ltnp | grep -E '15002|4040'`.

</details>

✅ **Expected:** `LISTEN ... *:15002 ... users:(("java",...))` and `*:4040`: a **Java** process (the server), not your Python kernel.

In [ ]:
# Your code here

❓ **Question:** Which process listens on 15002 and which on 4040? Is it your notebook's Python?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 5.3 Connect as a client

Create the **client session** with `SparkSession.builder.remote("sc://localhost:15002")`, print its **type** and version, and **repeat steps 2.1 and 2.3** on it: read `TrainBF` again into `df` and add the `revenue` column. Finally run the **channel query of 2.4** and check that you get the **same result**.

⚠️ The **server** reads the file, not your notebook: use the **absolute path** `/content/TrainBF.csv`.

<details><summary>💡 Hint</summary>

`from pyspark.sql import SparkSession`, `SparkSession.builder.remote(...).getOrCreate()`. The first connection can take ~10 s. `F` has to be imported again if you lost it: `from pyspark.sql import functions as F`.

</details>

✅ **Expected:** `<class 'pyspark.sql.connect.session.SparkSession'> 4.3.0` (note the `connect` in the class name), `rows: 250000` and the same three channels as in 2.4. The path is resolved **by the server process**, whose working directory is not `/content`.

In [ ]:
# Your code here

❓ **Question:** Which class is `spark` now? Why does a relative path like `TrainBF.csv` fail or depend on the folder where the server was started?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 5.4 What the client cannot do

Try to get the `sparkContext` of the Connect session and to build an RDD with `spark.range(5).rdd`. Catch the exception with `try/except` and print its type and the first 150 characters of the message.

<details><summary>💡 Hint</summary>

`try: spark.sparkContext` ... `except Exception as e: print(type(e).__name__, '-', str(e)[:150])`.

</details>

✅ **Expected:** Both lines fail with a *not implemented / not supported in Spark Connect* error: the client has no JVM and no `SparkContext`, so RDDs (Part 1) do not exist; you use **DataFrames and SQL** (and `df.collect()`, `toPandas()`...).

In [ ]:
# Your code here

❓ **Question:** Why does the Connect client reject both? Which part of this session (Part 1) could you not run on it? What do you use instead of the RDD API?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

💡 *Tip:* if you did Part 4, run its cell again now: the new server has its own UI on port 4040 and it has a **Connect** tab with the sessions and the requests of your client.

# Part 6 – 🧩 Final exercise: the report for MediaMas

The management of MediaMas wants the numbers of BlacFraidi. You work now on the **Connect session** (`spark`, `df` and `F` of step 5.3): the code is exactly the same as in classic mode.

## 6.1 The five star products

For each `product_id` compute the number of order lines (`orders`), the units sold (`units`) and the total `revenue` (rounded to 2 decimals). Show the **5 products with the highest revenue**.

<details><summary>💡 Hint</summary>

`groupBy('product_id').agg(...).orderBy(F.desc('revenue')).limit(5)`. Add `F.first('category')` if you want the category.

</details>

✅ **Expected:** A table of 5 rows. All five star products are `Electronics`: with few units they earn the most because they are expensive. The best one has more than 600 000 € of revenue.

In [ ]:
# Your code here

❓ **Question:** Is the product with the highest revenue also the one with the most units? Which category are they?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 6.2 Who returns what?

Returns cost money. Compute the **return rate** (`returned` is 0/1, so its average is the rate; show it as a percentage rounded to 1 decimal) for every **(channel, category)** pair and show the **5 pairs with the highest rate**, together with the number of order lines of each pair.

<details><summary>💡 Hint</summary>

`groupBy('channel', 'category').agg(F.round(F.avg('returned') * 100, 1).alias('return_pct'), F.count('*').alias('orders')).orderBy(F.desc('return_pct')).limit(5)`.

</details>

✅ **Expected:** The top: `App`-`Fashion` (22.6 %) and `Online`-`Fashion` (21.8 %), then `Online`/`App`-`Electronics` (≈ 14 %) and `Online`-`Sports` (11.4 %). **Both** things matter: `Fashion` (clothes do not fit) and the **remote** channels (you cannot try before you buy): in `Store` the rates are much lower.

In [ ]:
# Your code here

❓ **Question:** Which combination returns most? Is it the channel, the category or both that explains it? What would you do about it?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 6.3 The busiest moments

Which are the **5 busiest slots** of the campaign? Group by `order_date` **and** `order_hour`, add up the revenue and show the 5 best, in SQL this time (register the Connect DataFrame as the view `orders` first: the old view belonged to the classic session).

<details><summary>💡 Hint</summary>

`df.createOrReplaceTempView('orders')` and `SELECT order_date, order_hour, CAST(ROUND(SUM(revenue)) AS BIGINT) AS revenue FROM orders GROUP BY order_date, order_hour ORDER BY revenue DESC LIMIT 5`.

</details>

✅ **Expected:** The first slot is `2026-11-26` at **hour 0** (≈ 716 000 €: the midnight opening of the offers), followed by hours 21, 20, 22 and 19 of the same Friday. It matches the heat map.

In [ ]:
# Your code here

❓ **Question:** Does it match the heat map of Part 2? Why would MediaMas want to know this before next year's campaign?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 6.4 Save the report in Parquet

Save the whole `df` in **Parquet** format in the folder `bf_parquet`, **partitioned by `channel`** (`write.partitionBy('channel').parquet(...)`, mode `overwrite`). Then list the folder and compare the **size on disk** of the Parquet folder with that of the original `TrainBF.csv` (`du -sh`).

<details><summary>💡 Hint</summary>

`df.write.mode('overwrite').partitionBy('channel').parquet('/content/bf_parquet')` (absolute path: the **server** writes it); then `!ls /content/bf_parquet` and `!du -sh /content/bf_parquet /content/TrainBF.csv`.

</details>

✅ **Expected:** One sub-folder per channel (`channel=Online`, `channel=App`, `channel=Store`) plus `_SUCCESS`, and a total size of a small fraction of the CSV (≈ 4.6 MB against 23 MB of the CSV). Parquet is **columnar** and compressed; a query that only needs some columns or only the `Store` channel reads just those files (*column pruning* and *partition pruning*).

In [ ]:
# Your code here

❓ **Question:** What does the folder look like? Why is Parquet smaller and faster than CSV for this kind of query? What is the advantage of partitioning by `channel`?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### Your result

Paste the 5 rows of step 5.1 in the box below.

✍️ **Your result** *(double-click, then paste the 5 rows below, one per line inside this quote)*:

> 

# Part 7 – Clean up

## 7.1 Stop the client

Stop the Connect session (the client).

<details><summary>💡 Hint</summary>

`spark.stop()`.

</details>

✅ **Expected:** `client stopped`.

In [ ]:
# Your code here

## 7.2 Stop the server

Stop the **server** with `stop-connect-server.sh` (in the same `sbin` folder as the start script) and check with `ss` that port 15002 is no longer in use.

<details><summary>💡 Hint</summary>

`%%bash`, define `SPARK_HOME` as in 5.2 and run `$SPARK_HOME/sbin/stop-connect-server.sh`.

</details>

✅ **Expected:** `port 15002 is free`.

In [ ]:
# Your code here

### Final questions

Answer each one in the box below it.

❓ **Question:** What is the difference between a *transformation* and an *action*? Give two examples of each and say what *lazy evaluation* buys you.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** Spark Connect or classic mode: what changes for the programmer, which one would you use to share a cluster among many analysts, and what can you **not** do with Connect?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** Write **three sentences for the management of MediaMas** with the main findings of your analysis (when, where and what do they sell, and what is the problem with returns).

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 